# Numerical Model of a PCM-Enhanced Brick
## Transient 1D Heat Conduction Aligned with Experimental Sensors

This notebook simulates transient one-dimensional heat conduction in a brick.

Two cases are compared:
1. Standard brick (constant thermal properties)
2. PCM-enhanced brick (effective heat capacity formulation)

The results directly reproduce the quantities measured experimentally:
- Bottom sensor temperature
- Mid-height sensor temperature
- Top sensor temperature
- Temporal evolution


## Physical Model

The governing equation is the 1D transient heat equation:

$$ ρ c_p \frac{∂T}{∂t} = k \frac{∂²T}{∂x²} $$

For the PCM-enhanced material, the heat capacity becomes temperature dependent:

$ c_p,eff(T) = c_p + \frac{L}{ΔT} $ (within phase change interval)

This captures latent heat absorption during melting without explicitly tracking phase boundaries.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# Geometry
L = 0.1
Nx = 50
dx = L / (Nx - 1)
x = np.linspace(0, L, Nx)

In [3]:
# Time parameters
dt = 0.05
Nt = 2000
time = np.arange(Nt) * dt

In [4]:
# Material properties (brick)
rho = 1800      # kg/m3
k = 0.5         # W/mK
cp = 800        # J/kgK

# PCM properties (paraffin approx.)
L_latent = 150000   # J/kg
T_melt = 330        # K
delta_T = 5         # melting interval width

In [5]:
def cp_effective(T):
    cp_eff = np.full_like(T, cp)
    mask = (T > T_melt - delta_T/2) & (T < T_melt + delta_T/2)
    cp_eff[mask] += L_latent / delta_T
    return cp_eff

In [6]:
# Initial and boundary conditions
T0 = 350   # bottom temperature (heat source)
Tf = 300   # top temperature

T_const = np.ones(Nx) * Tf
T_pcm = np.ones(Nx) * Tf

T_const[0] = T0
T_pcm[0] = T0

In [7]:
# Sensor-equivalent positions
idx_bottom = 1
idx_mid = Nx // 2
idx_top = Nx - 2

In [8]:
# Storage arrays
bottom_const, mid_const, top_const = [], [], []
bottom_pcm, mid_pcm, top_pcm = [], [], []

In [9]:
# Time integration
for n in range(Nt):
    Tn_const = T_const.copy()
    Tn_pcm = T_pcm.copy()

    for i in range(1, Nx-1):
        
        # Constant material
        alpha_const = k / (rho * cp)
        T_const[i] = Tn_const[i] + alpha_const * dt / dx**2 * (
            Tn_const[i+1] - 2*Tn_const[i] + Tn_const[i-1]
        )
        
        # PCM material
        cp_eff = cp_effective(np.array([Tn_pcm[i]]))[0]
        alpha_pcm = k / (rho * cp_eff)
        T_pcm[i] = Tn_pcm[i] + alpha_pcm * dt / dx**2 * (
            Tn_pcm[i+1] - 2*Tn_pcm[i] + Tn_pcm[i-1]
        )

    # Boundary conditions
    T_const[0] = T0
    T_const[-1] = Tf
    
    T_pcm[0] = T0
    T_pcm[-1] = Tf

    # Save sensor temperatures
    bottom_const.append(T_const[idx_bottom])
    mid_const.append(T_const[idx_mid])
    top_const.append(T_const[idx_top])

    bottom_pcm.append(T_pcm[idx_bottom])
    mid_pcm.append(T_pcm[idx_mid])
    top_pcm.append(T_pcm[idx_top])